# Huấn luyện TwinLiteNetPlus phân vùng mặt đường
## 1. Mục tiêu
Huấn luyện lại hai lớp `background` (nền) và `road` (mặt đường) trên Colab GPU. Đầu vào NCHW (thứ tự batch, kênh, cao, rộng), RGB, 192×320; đầu ra logits (điểm chưa chuẩn hóa) hai kênh. Repo không kèm bản ghi cấu hình huấn luyện của model trong `models/`, nên lần train mới có thể cho trọng số và chất lượng khác model đó.

Hợp đồng lấy từ `models/twinlitenetplus_192x320_road_info.json`; dữ liệu xem `docs/datasets.md`. Biến thể small và nhánh SingleLiteNetPlus được suy từ tên nút, kích thước trọng số và số khối của ONNX hiện có. Siêu tham số train dưới đây là **giá trị mặc định**, không phải cấu hình đã tạo model trong `models/`.

**Cách dùng nhanh:** đặt hai tệp `ucr_road.zip` và `split.csv` vào `MyDrive/UIT_Car_Racing/Vong3/data` trên Google Drive, chọn Runtime GPU, rồi chạy lần lượt các ô. Tập train/valid/test lấy theo cột `split` của `split.csv`, không theo thư mục trong zip.

## 2. Môi trường và cấu hình duy nhất
Chọn Runtime → Change runtime type → GPU. `TWINLITE_COMMIT` ghim commit của repo nguồn có API `SingleLiteNetPlus` dùng ở mục 4; đổi commit thì chạy lại ô dò API. `DRIVE_DIR` là thư mục Drive chứa zip dữ liệu và `split.csv` (tên mặc định `ucr_road.zip`, `split.csv`; nếu không thấy `ucr_road.zip` mà thư mục có đúng một zip có chữ `road` trong tên thì dùng zip đó). `REF_DIR` tùy chọn, chứa các cặp tensor tham chiếu.

Mọi siêu tham số nằm trong ô này. Không đổi kích thước, chuẩn hóa hoặc opset nếu cần giữ hợp đồng suy luận. Cài phiên bản ghim cần Internet; chạy trên runtime mới. Repo nguồn: https://github.com/chequanghuy/TwinLiteNetPlus ; [LICENSE upstream](https://github.com/chequanghuy/TwinLiteNetPlus/blob/main/LICENSE) ghi MIT License. Ô clone in LICENSE tại chính commit đã chọn để đối chiếu trước khi sử dụng.

In [ ]:
TWINLITE_COMMIT = "90f1b8695ae311d5123b05f8534b2e11e42499d2"  # commit ghim của repo nguồn
DRIVE_DIR = "/content/drive/MyDrive/UIT_Car_Racing/Vong3/data"  # thư mục Drive chứa zip dữ liệu + split.csv
ZIP_NAME = "ucr_road.zip"  # zip dữ liệu COCO polygon
SPLIT_NAME = "split.csv"  # bảng chia tập (cột file, split, group_id)
REF_DIR = ""  # Để trống nếu không có bộ tensor tham chiếu.
CFG = dict(
    variant="small", seed=42, epochs=100, batch_size=16, lr=1e-3,
    optimizer="AdamW", weight_decay=1e-4, loss="CrossEntropyLoss",
    flip_probability=0.5, brightness=0.25, contrast=0.20,
    workers=2, amp=True, height=192, width=320, classes=2,
    scale=1 / 255, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
    opset=12, ort_atol=1e-4, ort_rtol=1e-3,
)
PACKAGES = ["torch==2.5.1", "numpy==1.26.4", "opencv-python-headless==4.10.0.84",
            "matplotlib==3.9.2", "onnx==1.17.0", "onnxruntime==1.20.1"]

assert (CFG["height"], CFG["width"], CFG["classes"], CFG["opset"]) == (192, 320, 2, 12), "Giữ hợp đồng tensor cố định"
assert CFG["epochs"] > 0 and CFG["batch_size"] > 1 and CFG["lr"] > 0
assert 0 <= CFG["flip_probability"] <= 1 and CFG["workers"] >= 0


In [ ]:
import subprocess, sys, re
subprocess.run(["nvidia-smi"], check=True, timeout=30)
subprocess.run([sys.executable, "-m", "pip", "install", *PACKAGES], check=True, timeout=900)
import torch
assert torch.cuda.is_available(), "Cần Colab GPU và torch CUDA hoạt động"
print("torch", torch.__version__, "GPU", torch.cuda.get_device_name(0))
assert re.fullmatch(r"[0-9a-fA-F]{40}", TWINLITE_COMMIT), "Điền TWINLITE_COMMIT: hash commit 40 ký tự của repo nguồn"
from pathlib import Path
REPO = Path("/content/TwinLiteNetPlus")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/chequanghuy/TwinLiteNetPlus.git", str(REPO)], check=True, timeout=180)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", TWINLITE_COMMIT], check=True, timeout=60)
actual_commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_commit.lower() == TWINLITE_COMMIT.lower()
assert (REPO / "LICENSE").is_file(), "Đọc và đối chiếu giấy phép repo nguồn trước khi tiếp tục"
print((REPO / "LICENSE").read_text())
sys.path.insert(0, str(REPO))


## 3. Dữ liệu COCO polygon và `split.csv`
Zip chứa ảnh và nhãn COCO polygon (`_annotations.coco.json` trong từng thư mục). Notebook **không** dùng cách chia thư mục train/valid/test của zip: nó gom ảnh của mọi thư mục rồi chia lại đúng theo cột `split` trong `split.csv` (chia theo lượt thu, không còn lượt nào nằm ở nhiều tập). Tên ảnh trong cột `file` phải trùng `file_name` trong JSON; thiếu, trùng hoặc có nhóm lượt thu (`group_id`) nằm ở nhiều tập thì notebook dừng. Category gốc ID 0 là lớp cha, không có annotation; không được hiểu ID 0 là pixel nền. Mọi annotation thuộc category không mang tên nền được tô road=1; nền là phần ảnh không được polygon phủ. Ô đọc dừng nếu ID 0 có annotation để người dùng kiểm nhãn.

Polygon (đa giác) được tô ở kích thước ảnh gốc rồi resize mask bằng nearest-neighbor (láng giềng gần nhất). Ảnh được resize toàn bộ bằng INTER_LINEAR, không crop hoặc padding; đổi BGR→RGB, nhân 1/255 rồi chuẩn hóa ImageNet ngoài model. Augmentation (tăng cường dữ liệu) chỉ áp dụng train: lật ngang đồng thời ảnh/mask và đổi độ sáng/tương phản ảnh. Đây là mặc định mới.

In [ ]:
import json, zipfile, random, hashlib, shutil, csv
import numpy as np
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict
random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
RESULTS = Path("/content/results")
RESULTS.mkdir(exist_ok=True)

from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
assert DRIVE.is_dir(), f"Không thấy thư mục {DRIVE}. Tạo MyDrive/UIT_Car_Racing/Vong3/data và tải zip + split.csv lên đó."
zip_path = DRIVE / ZIP_NAME
if not zip_path.is_file():
    found = [p for p in DRIVE.glob("*.zip") if "road" in p.name.lower()]
    assert len(found) == 1, f"Không thấy {ZIP_NAME}. Các zip trong thư mục: {[p.name for p in DRIVE.glob('*.zip')]}"
    zip_path = found[0]
split_path = DRIVE / SPLIT_NAME
assert split_path.is_file(), f"Không thấy {split_path}. Tải split.csv lên cùng thư mục với zip."
print("Zip:", zip_path.name, "|", zip_path.stat().st_size, "byte | split:", split_path.name)

DATA_ROOT = Path("/content/road_dataset")
assert not DATA_ROOT.exists(), "Thư mục dữ liệu đã tồn tại; dùng runtime mới để tránh trộn dữ liệu"
local_zip = Path("/content") / zip_path.name
shutil.copyfile(zip_path, local_zip)
with zipfile.ZipFile(local_zip) as archive:
    for item in archive.infolist():
        target = (DATA_ROOT / item.filename).resolve()
        assert target.is_relative_to(DATA_ROOT.resolve()), "Đường dẫn zip vượt thư mục dữ liệu"
        assert (item.external_attr >> 16) & 0o170000 != 0o120000, "Không nhận symlink trong zip"
    archive.extractall(DATA_ROOT)


In [ ]:
def polygon_to_mask(height, width, annotations, categories):
    mask = np.zeros((height, width), dtype=np.uint8)
    names = {int(c["id"]): c["name"].strip().lower() for c in categories}
    for ann in annotations:
        cid = int(ann["category_id"])
        assert cid in names, "Category không khai báo"
        if names[cid] in ("background", "nền"):
            continue
        polygons = ann["segmentation"]
        assert isinstance(polygons, list) and polygons, "Chỉ nhận COCO polygon, không nhận RLE"
        for polygon in polygons:
            points = np.asarray(polygon, dtype=np.float64)
            assert points.size >= 6 and points.size % 2 == 0 and np.isfinite(points).all(), "Polygon không hợp lệ"
            points = np.rint(points.reshape(-1, 2)).clip(-1e6, 1e6).astype(np.int32)
            cv2.fillPoly(mask, [points], 1)
    return mask

def preprocess_bgr(bgr):
    assert bgr is not None and bgr.ndim == 3 and bgr.shape[2] == 3
    small = cv2.resize(bgr, (CFG["width"], CFG["height"]), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    x = rgb.astype(np.float32) * np.float32(CFG["scale"])
    x = (x - np.asarray(CFG["mean"], np.float32)) / np.asarray(CFG["std"], np.float32)
    return np.ascontiguousarray(x.transpose(2, 0, 1), dtype=np.float32)


In [ ]:
from torch.utils.data import Dataset, DataLoader

def load_split_csv(path):
    with open(path, encoding="utf-8-sig", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert rows, "split.csv rỗng"
    for column in ("file", "split"):
        assert column in rows[0], "split.csv thiếu cột " + column
    names = [r["file"] for r in rows]
    assert len(set(names)) == len(names), "split.csv có tên file trùng"
    bad = {r["split"] for r in rows} - {"train", "valid", "test"}
    assert not bad, "Giá trị split không hợp lệ: " + str(bad)
    if "group_id" in rows[0]:
        seen = defaultdict(set)
        for r in rows:
            if r["group_id"]:
                seen[r["group_id"]].add(r["split"])
        leaks = [g for g, s in seen.items() if len(s) > 1]
        assert not leaks, f"{len(leaks)} nhóm lượt thu nằm ở nhiều tập: {leaks[:5]}"
    return rows

def build_index(root):
    index, categories, key0 = {}, None, None
    json_files = sorted(root.rglob("_annotations.coco.json"))
    assert json_files, "Zip không có _annotations.coco.json"
    for json_path in json_files:
        coco = json.loads(json_path.read_text(encoding="utf-8"))
        key = [(int(c["id"]), c["name"]) for c in coco["categories"]]
        if key0 is None:
            key0, categories = key, coco["categories"]
        assert key == key0, "Các JSON có categories khác nhau"
        ids = {im["id"] for im in coco["images"]}
        assert len(ids) == len(coco["images"]) and ids, "ID ảnh trùng hoặc JSON rỗng: " + str(json_path)
        annotations = defaultdict(list)
        for ann in coco["annotations"]:
            assert ann["image_id"] in ids
            assert int(ann["category_id"]) != 0, "Category cha ID 0 có annotation: kiểm lại nhãn"
            annotations[ann["image_id"]].append(ann)
        for im in coco["images"]:
            assert im["file_name"] not in index, "Ảnh trùng tên giữa các thư mục: " + im["file_name"]
            index[im["file_name"]] = dict(root=json_path.parent, im=im, anns=annotations[im["id"]])
    return index, categories

SPLIT_ROWS = load_split_csv(split_path)
INDEX, CATEGORIES = build_index(DATA_ROOT)
missing = [r["file"] for r in SPLIT_ROWS if r["file"] not in INDEX]
assert not missing, f"{len(missing)} file trong split.csv không có trong zip, ví dụ: {missing[:3]}"
for r in SPLIT_ROWS:
    im = INDEX[r["file"]]["im"]
    if r.get("width") and r.get("height"):
        assert (int(r["width"]), int(r["height"])) == (im["width"], im["height"]), "Kích thước khác split.csv: " + r["file"]
unused = len(INDEX) - len(SPLIT_ROWS)
if unused:
    print("Cảnh báo:", unused, "ảnh trong zip không có trong split.csv, bỏ qua")
print("Tổng ảnh theo split.csv:", {s: sum(r["split"] == s for r in SPLIT_ROWS) for s in ("train", "valid", "test")})
SPLIT_SHA256 = hashlib.sha256(split_path.read_bytes()).hexdigest()

class RoadDataset(Dataset):
    def __init__(self, split, augment=False):
        self.records = [INDEX[r["file"]] for r in SPLIT_ROWS if r["split"] == split]
        assert self.records, "Split rỗng: " + split
        self.categories = CATEGORIES
        self.augment = augment
        print(split, len(self.records), "ảnh; categories:", self.categories)
    def __len__(self):
        return len(self.records)
    def raw(self, index):
        rec = self.records[index]
        im = rec["im"]
        path = (rec["root"] / im["file_name"]).resolve()
        assert path.is_relative_to(rec["root"].resolve()), "Ảnh vượt thư mục dữ liệu"
        bgr = cv2.imread(str(path))
        assert bgr is not None, "Không đọc được ảnh: " + str(path)
        assert bgr.shape[:2] == (im["height"], im["width"]), "Kích thước ảnh khác JSON"
        mask = polygon_to_mask(im["height"], im["width"], rec["anns"], self.categories)
        return bgr, mask
    def __getitem__(self, index):
        bgr, mask = self.raw(index)
        if self.augment:
            if random.random() < CFG["flip_probability"]:
                bgr, mask = cv2.flip(bgr, 1), cv2.flip(mask, 1)
            alpha = random.uniform(1 - CFG["contrast"], 1 + CFG["contrast"])
            beta = random.uniform(-CFG["brightness"], CFG["brightness"]) * 255
            bgr = np.clip(bgr.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        x = preprocess_bgr(bgr)
        y = cv2.resize(mask, (CFG["width"], CFG["height"]), interpolation=cv2.INTER_NEAREST)
        return torch.from_numpy(x), torch.from_numpy(y.astype(np.int64))

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

datasets = {s: RoadDataset(s, augment=s == "train") for s in ("train", "valid", "test")}
generator = torch.Generator().manual_seed(CFG["seed"])
loaders = {s: DataLoader(ds, batch_size=CFG["batch_size"], shuffle=s == "train",
                        num_workers=CFG["workers"], pin_memory=True, worker_init_fn=seed_worker,
                        generator=generator) for s, ds in datasets.items()}
fig, axes = plt.subplots(4, 2, figsize=(10, 12))
for i in range(4):
    bgr, mask = datasets["train"].raw(i % len(datasets["train"]))
    axes[i, 0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    axes[i, 1].imshow(mask, vmin=0, vmax=1, cmap="gray")
    for ax in axes[i]: ax.axis("off")
fig.tight_layout()
fig.savefig(RESULTS / "data_preview.png")
plt.show()


## 4. Mô hình
Dò chữ ký tại commit đã clone, rồi dùng `SingleLiteNetPlus(args.config)` của repo nguồn: nhánh phân vùng đơn của họ TwinLiteNetPlus. Không dùng loss hoặc dataloader upstream. Kiểm tra tensor đầu ra trước khi train; dừng nếu API hoặc hình dạng khác. Không thêm softmax vào model; argmax trực tiếp trên hai kênh, hòa chọn nền.

In [ ]:
import inspect
from types import SimpleNamespace
from model.model import SingleLiteNetPlus
from model import config as upstream_config
print("Chữ ký:", inspect.signature(SingleLiteNetPlus))
print("Cấu hình upstream:", upstream_config.sc_ch_dict[CFG["variant"]])
model = SingleLiteNetPlus(SimpleNamespace(config=CFG["variant"])).cuda()
model.eval()
with torch.no_grad():
    probe = model(torch.zeros(1, 3, CFG["height"], CFG["width"], device="cuda"))
assert isinstance(probe, torch.Tensor) and tuple(probe.shape) == (1, 2, CFG["height"], CFG["width"])
assert torch.isfinite(probe).all()
print("Số tham số PyTorch:", sum(p.numel() for p in model.parameters()))


## 5. Huấn luyện
AMP (tính toán hỗn hợp độ chính xác) giảm bộ nhớ GPU. Dùng CrossEntropyLoss và AdamW với learning rate (tốc độ học) cố định; không có bản ghi loss/optimizer/lịch học của model trong `models/`. Lưu checkpoint (trạng thái trọng số) tốt nhất theo mIoU valid, không dùng test để chọn checkpoint. mIoU là trung bình IoU các lớp có hợp khác 0; chỉ số không xác định ghi null khi xuất JSON.

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
assert CFG["optimizer"] == "AdamW" and CFG["loss"] == "CrossEntropyLoss"
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"])

def scores(matrix):
    m = matrix.astype(np.float64)
    tp = np.diag(m)
    union = m.sum(0) + m.sum(1) - tp
    iou = np.divide(tp, union, out=np.full(2, np.nan), where=union > 0)
    precision = tp[1] / m[:, 1].sum() if m[:, 1].sum() else float("nan")
    recall = tp[1] / m[1].sum() if m[1].sum() else float("nan")
    return dict(iou_background=float(iou[0]), iou_road=float(iou[1]), miou=float(np.nanmean(iou)),
                precision_road=float(precision), recall_road=float(recall))

def run_epoch(loader, training):
    model.train(training)
    total_loss, count = 0.0, 0
    matrix = np.zeros((2, 2), np.int64)
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
            if training: optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=CFG["amp"]):
                logits = model(x)
                loss = criterion(logits, y)
            assert torch.isfinite(loss), "Loss không hữu hạn"
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * len(x)
            count += len(x)
            indices = (y * 2 + logits.detach().argmax(1)).reshape(-1)
            matrix += torch.bincount(indices, minlength=4).reshape(2, 2).cpu().numpy()
    return dict(loss=total_loss / count, **scores(matrix)), matrix

history = []
best_miou = -1.0
for epoch in range(1, CFG["epochs"] + 1):
    train_metrics, _ = run_epoch(loaders["train"], True)
    valid_metrics, _ = run_epoch(loaders["valid"], False)
    history.append(dict(epoch=epoch, train=train_metrics, valid=valid_metrics))
    print(epoch, "train", train_metrics, "valid", valid_metrics)
    if valid_metrics["miou"] > best_miou:
        best_miou = valid_metrics["miou"]
        torch.save(dict(model=model.state_dict(), config=CFG, epoch=epoch,
                        valid=valid_metrics, upstream_commit=actual_commit), RESULTS / "best.pt")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split in ("train", "valid"):
    axes[0].plot([h["epoch"] for h in history], [h[split]["loss"] for h in history], label=split)
    axes[1].plot([h["epoch"] for h in history], [h[split]["miou"] for h in history], label=split)
for ax, title in zip(axes, ("Loss", "mIoU")):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
fig.tight_layout()
fig.savefig(RESULTS / "training_curves.png")
plt.show()


## 6. Đánh giá test
Nạp trọng số tốt nhất theo valid, đánh giá FP32 không augmentation. In IoU từng lớp, mIoU, precision (độ chính xác dự đoán dương) và recall (độ bao phủ) của road. Lưới gồm ảnh, nhãn và dự đoán. Tập test là các lượt thu không xuất hiện ở train và valid (theo `split.csv`), nhưng vẫn chỉ là một số lượt thu trên sa bàn nên chưa đại diện đầy đủ các điều kiện chạy xe.

In [ ]:
checkpoint = torch.load(RESULTS / "best.pt", map_location="cuda", weights_only=True)
model.load_state_dict(checkpoint["model"])
model.eval()
training_amp = CFG["amp"]
CFG["amp"] = False
try:
    test_metrics, confusion = run_epoch(loaders["test"], False)
finally:
    CFG["amp"] = training_amp
print("Test:", test_metrics, "ma trận nhầm lẫn (hàng=nhãn, cột=dự đoán):", confusion)
fig, axes = plt.subplots(4, 3, figsize=(12, 10))
for i in range(4):
    bgr, mask = datasets["test"].raw(i % len(datasets["test"]))
    x = torch.from_numpy(preprocess_bgr(bgr)[None]).cuda()
    with torch.no_grad(): prediction = model(x).argmax(1)[0].cpu().numpy()
    axes[i, 0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    axes[i, 1].imshow(mask, cmap="gray", vmin=0, vmax=1)
    axes[i, 2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
    for ax, title in zip(axes[i], ("Ảnh", "Nhãn", "Dự đoán")):
        ax.set_title(title)
        ax.axis("off")
fig.tight_layout()
fig.savefig(RESULTS / "test_predictions.png")
plt.show()


## 7. Xuất ONNX và đối chiếu
Xuất FP32, opset 12, không dynamic, tên input/output cố định, hình dạng 1×3×192×320 → 1×2×192×320. Không đưa chuẩn hóa hoặc softmax vào graph. So sánh ONNX Runtime với PyTorch trên toàn bộ test và input tham chiếu nếu có: sai khác tuyệt đối lớn nhất, tỉ lệ pixel argmax trùng; kiểm allclose theo dung sai trong cấu hình. Kết quả chỉ xác nhận bản xuất, chưa xác nhận TensorRT.

`REF_DIR` chứa `NNN_input.npy` và `NNN_logits.npy`: input đã chuẩn hóa, không chuẩn hóa lại. Tỉ lệ dự đoán model mới trùng model cũ chỉ để tham khảo, không phải tiêu chí đạt hay IoU với nhãn đúng.

In [ ]:
import onnx
import onnxruntime as ort
onnx_path = RESULTS / "twinlitenetplus_192x320.onnx"
model = model.cpu().float().eval()
dummy = torch.zeros(1, 3, CFG["height"], CFG["width"], dtype=torch.float32)
torch.onnx.export(model, dummy, str(onnx_path), input_names=["input"], output_names=["output"],
                  opset_version=CFG["opset"], dynamic_axes=None, do_constant_folding=True, dynamo=False)
graph = onnx.load(str(onnx_path))
onnx.checker.check_model(graph)
assert next(o.version for o in graph.opset_import if o.domain == "") == 12
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
assert session.get_inputs()[0].name == "input" and session.get_inputs()[0].shape == [1, 3, 192, 320]
assert session.get_outputs()[0].name == "output" and session.get_outputs()[0].shape == [1, 2, 192, 320]
max_error, matched, pixels, comparison_count = 0.0, 0, 0, 0
reference_comparison = []

def compare_export(x):
    global max_error, matched, pixels, comparison_count
    assert x.shape == (1, 3, 192, 320) and x.dtype == np.float32 and np.isfinite(x).all()
    with torch.no_grad(): pt = model(torch.from_numpy(x)).numpy()
    exported = session.run(["output"], {"input": x})[0]
    assert np.isfinite(pt).all() and np.isfinite(exported).all()
    max_error = max(max_error, float(np.max(np.abs(exported - pt))))
    matched += int(np.sum(exported.argmax(1) == pt.argmax(1)))
    pixels += int(pt.shape[0] * pt.shape[2] * pt.shape[3])
    comparison_count += 1
    np.testing.assert_allclose(exported, pt, atol=CFG["ort_atol"], rtol=CFG["ort_rtol"])
    return pt

for i in range(len(datasets["test"])):
    bgr, _ = datasets["test"].raw(i)
    compare_export(preprocess_bgr(bgr)[None])
if REF_DIR:
    reference_files = sorted(Path(REF_DIR).glob("*_input.npy"))
    assert reference_files, "REF_DIR không có tensor input"
    for path in reference_files:
        x = np.load(path, allow_pickle=False)
        old = np.load(path.with_name(path.name.replace("_input.npy", "_logits.npy")), allow_pickle=False)
        if old.shape == (2, 192, 320): old = old[None]
        assert old.shape == (1, 2, 192, 320) and np.isfinite(old).all()
        new = compare_export(x)
        reference_comparison.append(dict(input=path.name, argmax_match_old=float(np.mean(new.argmax(1) == old.argmax(1)))))
    print("Tham khảo model cũ:", reference_comparison)
export_comparison = dict(max_absolute_error=max_error, argmax_match=matched / pixels, inputs=comparison_count)
print("ONNX Runtime / PyTorch:", export_comparison)
info = dict(task="segmentation", architecture="twinlitenetplus", class_names=["background", "road"],
            num_classes=2, mask_values=dict(background=0, road=1), source_image_hw=[720,1280],
            crop_y=[0,720], crop_x=[0,1280], resize="cv2.resize INTER_LINEAR (full -> input)",
            input_name="input", input_shape=[1,3,192,320], input_hw=[192,320], input_dtype="float32",
            color_order="RGB", layout="NCHW", normalization=dict(scale=CFG["scale"], mean=CFG["mean"], std=CFG["std"]),
            imagenet_normalization_in_model=False, preprocessing_in_model=False, output_name="output",
            output_shape=[1,2,192,320], output_type="logits", activation="softmax_over_channels_argmax",
            road_rule="output[0,1] > output[0,0]", dynamic=False, fake_data=False, opset=12,
            onnx_sha256=hashlib.sha256(onnx_path.read_bytes()).hexdigest())
(RESULTS / "twinlitenetplus_192x320_road_info.json").write_text(json.dumps(info, indent=2), encoding="utf-8")


## 8. Lưu kết quả
Chỉ ghi số đo thật sau khi chạy. `results/` chứa checkpoint tốt nhất, ONNX, siêu dữ liệu, biểu đồ và JSON chỉ số. JSON lưu cấu hình, commit nguồn, số ảnh từng split và lịch sử train để lần chạy sau đối chiếu. Tải results.zip trước khi kết thúc runtime.

In [ ]:
def json_safe(value):
    if isinstance(value, dict): return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, list): return [json_safe(v) for v in value]
    if isinstance(value, float) and not np.isfinite(value): return None
    return value
metrics = dict(config=CFG, upstream_commit=actual_commit, packages=PACKAGES,
               best_epoch=checkpoint["epoch"], best_valid=checkpoint["valid"],
               split_images={s: len(ds) for s, ds in datasets.items()}, test=test_metrics,
               confusion_matrix=confusion.tolist(), history=history, onnx_comparison=export_comparison,
               old_model_reference=reference_comparison,
               split_source=dict(file=SPLIT_NAME, sha256=SPLIT_SHA256, zip=zip_path.name),
               defaults_note="Siêu tham số mặc định của notebook, không phải cấu hình đã tạo model trong models/")
(RESULTS / "twinlite_metrics.json").write_text(json.dumps(json_safe(metrics), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
archive_path = shutil.make_archive("/content/results", "zip", RESULTS)
from google.colab import files
files.download(archive_path)


## 9. Hạn chế
- Không có công thức train của model trong `models/`; cùng hợp đồng tensor không đồng nghĩa cùng trọng số hoặc chất lượng.
- Phải điền commit nguồn đã kiểm tra; ô dò API sẽ dừng nếu repo tại commit đó khác API đã mô tả.
- Chia tập theo `split.csv` (nhóm theo lượt thu nên không có lượt nào nằm ở hai tập). Ảnh không có số lượt trong tên (nhóm `park_`, `road_them_`, `image`, `h9342_`) được gom theo giả định (khối 10 số liền nhau), nên chưa chắc không còn khung gần nhau ở hai tập. Model đã phát hành trong `models/` được train trên cách chia khác, nên IoU của lần train này không so trực tiếp được với model đó.
- Seed cố định giúp lặp lại nhưng khác GPU/CUDA có thể cho kết quả khác.
- Kiểm ONNX Runtime không thay thế build TensorRT 8.2 và đối chiếu trên Jetson, cũng không xác nhận độ an toàn hoặc chất lượng bám đường trên sa bàn.
- Notebook chưa được chạy train/forward/export trên GPU trong lần chuẩn bị này. Không có số liệu kết quả điền sẵn.